# BHSM Blackhole Tracker: conditional R1 response

The R1 extension uses one adopted scalar–tensor action and its metric response. Scalar flux provenance, worldtube radius and approximation budgets are explicit. The uniform bound restricts the admitted boundary-state norm. It does not select an astrophysical amplitude or a horizon solution.

This extends the same-potential BHSM interpretation. An ordinary envelope contribution already represented by a metric multipole is not added again. R1 `mass_msun` labels the action's metric/source mass; its total optical response includes the scalar term, so it is not automatically the ordinary point-exterior monopole. The Schwarzschild boundary conversion is not assumed for the modified metric. The metadata/reporting checks below do not test exclusive physical identification.

This notebook keeps the same declared synthetic measurement errors; it does not enlarge the physical state to manufacture detectability.

In [1]:
import json
import numpy as np
from bhtracker import fit
from bhtracker.examples import make_example
from bhtracker.physics_admission import assess_physics
dataset, truth = make_example('nearby_motion')
result = fit(dataset, family='r1', nuisance='shear', starts=2)
interpretation = result['bhsm_interpretation']
assert interpretation['framework'] == 'BHSM'
assert 'counted once' in interpretation['one_potential_rule']  # reporting contract
summary = {key: result[key] for key in ['status', 'estimates', 'conditional_envelope']}
summary['BHSM_same_response_interpretation'] = {key: interpretation[key] for key in ['model_role', 'one_potential_rule', 'conditional_extension', 'spherical_boundary']}
summary

{'status': 'TRACKED',
 'estimates': {'mass_msun': 99947.19835137356,
  'center_mas': [0.7920975656720222, -0.5965511898835554],
  'proper_motion_mas_yr': [0.0666675727060446, -0.03808274400762192],
  'shear': [0.00021199203311307368, 0.0003902972645727605],
  'compact_quad': [0.0, 0.0],
  'convergence': 0.0,
  'flux_ratio': 1.0},
 'conditional_envelope': {'status': 'UNRESOLVED',
  'boundary_amplitude_cap': 5.335514951059139e-21,
  'cap_definition': 'uniform sufficient retained-action admission bound; not a selected astrophysical amplitude',
  'rank': 2,
  'nuisance_rank': 25,
  'projected_singular_values_per_boundary_amplitude': [4364027.969635325,
   2297847.86555566],
  'unconstrained_local_amplitude_estimate': [3.7619381593567686e-07,
   -1.3832367234978532e-07],
  'unconstrained_estimate_inside_admitted_ball': False,
  'maximum_admitted_local_snr': 2.3284336478829534e-14,
  'worst_orientation_admitted_local_snr': 1.2260201641931554e-14,
  'all_admitted_states_below_unit_snr': True,

## Physical admission and canonical source pairing

In [2]:
admission = result['physics_admission']
{key: admission[key] for key in ['allowed_physics_models', 'boundary_state', 'source_pairing', 'approximation_budgets']}

{'allowed_physics_models': {'baseline_point_mass': True,
  'r1_spherical_conditional': True,
  'r1_boundary_conditional': True,
  'horizon_matched_r1': False},
 'boundary_state': {'amplitudes_plus_cross_axial': [0.0, 0.0, 0.0],
  'conormal_coefficients': [0.0, 0.0, 0.0],
  'amplitude_norm': 0.0,
  'impedance_l2': 0.011391433651418646,
  'energy_over_Mpl2_L0': 0.0,
  'eta': 0.0,
  'eta_definition': '6*||a||/(x0*|p0|)',
  'bound_kind': 'uniform charged-background relative bound',
  'amplitude_norm_max': 5.335514951059139e-21,
  'conormal_norm_max': 6.077916456114238e-23,
  'individual_sufficient_caps': {'eta': 5.335514951059139e-21,
   'relative_hessian': 4.883550057517992e-20,
   'spatial_principal': 1.2208875143794979e-20,
   'time_principal': 2.99055144356463e-20},
  'uniform_hessian_absolute_bound': 0.0,
  'uniform_hessian_relative_bound': 0.0,
  'uniform_spatial_principal_fraction_bound': 0.0,
  'uniform_time_principal_fraction_bound': 0.0,
  'uniform_nonlinear_remainder_over_princi

In [3]:
boundary = admission['boundary_state']
a_max = boundary['amplitude_norm_max']
q_max = boundary['impedance_l2'] * a_max
assert np.isclose(q_max, boundary['conormal_norm_max'], rtol=1e-12, atol=0)
{'Dirichlet_norm_cap': a_max, 'conormal_norm_cap': q_max,
 'horizon_matched': admission['horizon_matched_r1']}

{'Dirichlet_norm_cap': 5.335514951059139e-21,
 'conormal_norm_cap': 6.077916456114238e-23,
 'horizon_matched': False}

## Honest unresolved output
An unconstrained local coefficient that lies outside the admitted state ball is not a physical deformation measurement. The largest local SNR in the admitted ball tests whether even the most favorable admitted orientation could be resolved in this data design.

In [4]:
envelope = result['conditional_envelope']
assert envelope['status'] == 'UNRESOLVED'
assert envelope['all_admitted_states_below_unit_snr']
{key: envelope[key] for key in ['status', 'boundary_amplitude_cap',
 'maximum_admitted_local_snr', 'unconstrained_estimate_inside_admitted_ball']}

{'status': 'UNRESOLVED',
 'boundary_amplitude_cap': 5.335514951059139e-21,
 'maximum_admitted_local_snr': 2.3284336478829534e-14,
 'unconstrained_estimate_inside_admitted_ball': False}

## Reject an inadmissible state instead of converting noise into a discovery

In [5]:
bad_state = dict(result['_internal']['parameters'])
bad_state['boundary_amplitudes'] = [100 * a_max, 0, 0]
rejected = assess_physics(dataset, bad_state, result['branch_admission']['impact_pc'])
assert not rejected['allowed_physics_models']['r1_boundary_conditional']
rejected['reasons']

['state_within_sufficient_cap']

An unresolved conditional surface family does not remove the BHSM interpretation of the ordinary mass, location and translation. The product uses one optical response within its declared domain. Separate microscopic, horizon-charge or deformation-dynamics inference requires the corresponding source/matching relations; these are not conditions for retaining the ordinary track.